# Chapter 2 - Lab 5: Complaint reply drafting with the evaluator-optimizer pattern

**Evaluator-optimizer** uses two agents in a loop:
* a **generator** produces an answer;
* an **evaluator** critiques it and sends feedback back for revision,

until the evaluator is satisfied or a maximum number of iterations is reached.

*Finance version*: a news-search agent collected recent Reuters headlines; a judge agent checked the
links, dates and region.
*Customer-care version*: a **Reply Drafter** writes a response to a customer complaint; a **QA Evaluator**
checks it against what a contact-centre quality team checks:

1. every policy statement cites a knowledge-base article (`[KB-xxx]`): no invented policies;
2. every commitment has a concrete date or time frame;
3. no promise exceeds the agent's authority (for example credits above Rs 1,000 need a supervisor);
4. the tone is empathetic and addresses every issue the customer raised.

This is **LLM-as-a-judge**, mirroring how QA analysts score a sample of agent replies every day.

In [ ]:
%pip install -q openai-agents

In [ ]:
import os

try:
    from google.colab import userdata  # Colab: keys live under the key icon
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY') or ''
except ImportError:
    # Running locally: read keys from the environment or a .env file.
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass

assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first (Colab secret or .env)'

In [ ]:
import asyncio
from dataclasses import dataclass
from typing import Literal

from agents import Agent, ItemHelpers, Runner, TResponseInputItem, function_tool

## The knowledge base and a search tool (replaces the hosted web-search tool)

In [ ]:
# Mock knowledge base (KB) of NovaMart policies. In production: your help-centre / policy CMS.
KB_ARTICLES = {
    "KB-101": ("Return & replacement window",
               "Defective or damaged-on-arrival products can be replaced within 10 days of delivery. "
               "Change-of-mind returns: 7 days, not applicable to large appliances once installed."),
    "KB-102": ("Warranty repeat-failure rule",
               "If the same defect persists after 2 repair attempts within warranty, the customer is entitled "
               "to a replacement of the same model. Replacement delivery: 5-7 working days; Platinum priority "
               "dispatch within 72 hours when stock is available."),
    "KB-103": ("Goodwill credit authority",
               "Agents may offer up to Rs 1,000 goodwill credit. Rs 1,001-5,000 requires supervisor approval. "
               "Above Rs 5,000 requires CX-manager approval. Credits never exceed 15% of the order value."),
    "KB-104": ("Delivery delay compensation",
               "Orders delivered 3+ days after the promised date receive a Rs 250 voucher; 5+ days late receive "
               "a delivery-fee refund plus a Rs 500 voucher. Platinum members receive double the voucher value."),
    "KB-105": ("Product safety escalation",
               "Any report of smoke, burning smell, sparks, fire, electric shock or injury must be escalated to the "
               "Product Safety team within 1 hour. Advise the customer to switch off and unplug the product."),
}

In [ ]:
@function_tool
def search_policy_kb(query: str) -> str:
    """Search NovaMart's policy knowledge base. Returns matching articles with their KB IDs."""
    words = {w for w in query.lower().split() if len(w) > 3}
    hits = [(kb, title, body) for kb, (title, body) in KB_ARTICLES.items()
            if words & set((title + " " + body).lower().replace(",", " ").replace(".", " ").split())]
    hits = hits or [(kb, title, body) for kb, (title, body) in KB_ARTICLES.items()]
    return "\n".join(f"[{kb}] {title}: {body}" for kb, title, body in hits)

## The two agents

In [ ]:
reply_drafter = Agent(
    name="reply_drafter",
    instructions=(
        "You are a NovaMart customer-care agent writing a reply to a customer complaint. "
        "Always search the policy knowledge base first. Write a reply of at most 180 words. "
        "Cite the KB article ID in square brackets, e.g. [KB-102], after every policy statement. "
        "Give a concrete date or time frame for every commitment. "
        "Do not promise anything that exceeds an agent's authority without saying it needs approval. "
        "Address every issue the customer raised, with empathy."
    ),
    tools=[search_policy_kb],
    model="gpt-4o-mini",
)

@dataclass
class EvaluationFeedback:
    feedback: str
    score: Literal["successful", "unsuccessful"]

qa_evaluator = Agent(
    name="qa_evaluator",
    instructions=(
        "You are a contact-centre quality analyst. You receive a customer complaint and a draft reply. "
        "Check four criteria: (1) every policy statement cites a KB article ID like [KB-101]; "
        "(2) every commitment has a concrete date or time frame; (3) no promise exceeds agent authority "
        "(credits above Rs 1,000 need supervisor approval, above Rs 5,000 need CX-manager approval); "
        "(4) the reply is empathetic and addresses every issue the customer raised. "
        "Return score 'successful' only if all four criteria pass; otherwise 'unsuccessful' with specific feedback."
    ),
    output_type=EvaluationFeedback,
    model="gpt-4o-mini",
)

## The feedback loop

Termination: stop when the evaluator returns `successful` (convergence) or after `max_iteration = 4`
rounds (budget). Production systems always need the budget: an unbounded loop is an unbounded bill.

In [ ]:
async def main(complaint: str, max_iteration: int = 4) -> str:
    input_items: list[TResponseInputItem] = [{"content": complaint, "role": "user"}]
    latest_draft: str | None = None

    for iteration in range(1, max_iteration + 1):
        draft_result = await Runner.run(reply_drafter, input_items)
        input_items = draft_result.to_input_list()
        latest_draft = ItemHelpers.text_message_outputs(draft_result.new_items)
        print(f"\033[92m******** DRAFT {iteration} ********\033[0m\n{latest_draft}\n")

        eval_result = await Runner.run(qa_evaluator, f"COMPLAINT:\n{complaint}\n\nDRAFT REPLY:\n{latest_draft}")
        result: EvaluationFeedback = eval_result.final_output
        print(f"\033[94mQA score: {result.score}\nQA feedback: {result.feedback}\033[0m\n")

        if result.score == "successful":
            print("\033[92mQA passed: stopping the loop.\033[0m")
            break
        input_items.append({"content": f"QA feedback: {result.feedback}\nRevise the reply to address every point.",
                            "role": "user"})
    else:
        print("\033[91mReached max_iteration: route to a human supervisor.\033[0m")

    print("\n******** FINAL REPLY ********")
    print(latest_draft)
    return latest_draft

## Complaint 1: delayed delivery

In [ ]:
complaint_1 = """Order ORD-78120 (Zenith X1 phone) was promised for 20 October and it still has not arrived.
I am a Gold member. I want compensation and a firm delivery date. - Rahul Verma, C1002"""
final_reply = await main(complaint_1)

## Complaint 2: repeated repair failure, money and a social-media threat

In [ ]:
complaint_2 = """Two technician visits failed to fix my FrostLine refrigerator (order ORD-77812, Rs 62,990).
I threw away food worth Rs 4,000. I want a replacement AND a Rs 6,000 refund for my losses, otherwise
I will post about NovaMart on Twitter. - Priya Sharma, C1001 (Platinum)"""
final_reply = await main(complaint_2)

Watch the evaluator push back if the first draft promises Rs 6,000 outright (above agent authority,
see [KB-103]) or forgets the replacement timeline. Usually the loop converges in 1-3 iterations.

## Discussion
1. Today a QA team samples about 2% of interactions. What changes if an AI evaluator checks 100% *before* sending?
2. Should the evaluator use the same model as the drafter? (Hint: correlated blind spots.)
3. Which criteria are better checked by **code** (e.g., regex for `[KB-xxx]`, credit amount limits) than by an LLM?